<a href="https://colab.research.google.com/github/kc281/IA_CLASIFICADOR_DE_RESIDUOS/blob/main/PRUEBADATASET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from pathlib import Path

root = Path("/content/drive/MyDrive/DATASET_IA_standardized_256")
extensiones = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

counts = {}
for cls in sorted(os.listdir(root)):
    p = root / cls
    if p.is_dir():
        counts[cls] = sum(1 for f in p.rglob("*") if f.suffix.lower() in extensiones)

total = sum(counts.values())
print(f"{'Clase':<22}{'Imágenes':>10}{'%':>9}")
print("-" * 41)
for cls, n in counts.items():
    print(f"{cls:<22}{n:>10}{100*n/total:>8.1f}%")
print("-" * 41)
print(f"{'TOTAL':<22}{total:>10}{100.0:>8.1f}%")

Clase                   Imágenes        %
-----------------------------------------
biological                   699    11.7%
cardboard                   1411    23.6%
metal                        930    15.6%
paper                       1336    22.4%
plastic                     1597    26.7%
-----------------------------------------
TOTAL                       5973   100.0%


In [ ]:
import os, shutil
from pathlib import Path
from sklearn.model_selection import train_test_split

random_state = 42
root = Path("/content/drive/MyDrive/DATASET_IA_standardized_256")        # <-- ajusta
destino = Path("/content/drive/MyDrive/dataset_split")  # <-- ajusta
extensiones = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# 1) Recolectar rutas y etiquetas
files, labels = [], []
for cls in sorted(os.listdir(root)):
    carpeta = root / cls
    if carpeta.is_dir():
        for f in carpeta.iterdir():
            if f.suffix.lower() in extensiones:
                files.append(f)
                labels.append(cls)

# 2) Split estratificado 70/15/15
X_train, X_temp, y_train, y_temp = train_test_split(
    files, labels, test_size=0.30, stratify=labels, random_state=random_state)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=random_state)

# 3) Copiar a estructura train/val/test
for split_name, Xs, ys in [("train", X_train, y_train),
                            ("val",   X_val,   y_val),
                            ("test",  X_test,  y_test)]:
    for src, cls in zip(Xs, ys):
        out_dir = destino / split_name / cls
        out_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, out_dir / src.name)

# 4) Reporte de conteos
from collections import Counter
for nombre, ys in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{nombre:>6}: {dict(Counter(ys))}  (total={len(ys)})")

 train: {'plastic': 1118, 'paper': 935, 'biological': 489, 'metal': 651, 'cardboard': 988}  (total=4181)
   val: {'plastic': 240, 'cardboard': 211, 'metal': 140, 'paper': 200, 'biological': 105}  (total=896)
  test: {'metal': 139, 'paper': 201, 'cardboard': 212, 'plastic': 239, 'biological': 105}  (total=896)
